<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Coverage Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name','crop.id']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from coverage_functions.py**

### 🗺️ Configure extraction

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor
extractor = CoverageExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id"}

extractor.setup_coverage_parameters(
                                    vegetation_index= 'NDVI',
                                    start_date="2025-01-01",
                                    clear_cover_min=80,
                                    use_specific_date=False,
                                    filter="duplicate",
                                    delay=3,
                                    mask='auto',#Available masks (auto, native, ACM, ML). To configure.
                                    partial_frequency=20,
                                    exclude_columns=[],
                                    column_mapping=column_mapping,
                                    use_cache=True
) 

### 🗺️ Test functions

In [ ]:
test_entity = {
    "id": "test_001",
    "geometry": "POLYGON((2.2945 48.8584, 2.2955 48.8584, 2.2955 48.8594, 2.2945 48.8594, 2.2945 48.8584))"
}

#### API call

In [ ]:
print("\n--- Test: get_satellite_coverage_by_geometry ---")
try:
    raw_response = extractor.get_satellite_coverage_by_geometry(test_entity)
    print("✅ Raw API response received:")
    print(raw_response if isinstance(raw_response, dict) else raw_response[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")

#### Test get_satellite_coverage_by_geometry_safe

In [ ]:
print("\n--- Test: get_satellite_coverage_by_geometry_safe ---")
safe_result = extractor.get_satellite_coverage_by_geometry_safe(test_entity)
print(safe_result)

#### Test format_coverage_json

In [ ]:
print("\n--- Test: format_coverage_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_coverage_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_coverage_json: No valid data from API.")

### 🗺️ process_single_entity

In [ ]:
# Import your function

import pandas as pd
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))"
})

result = extractor.process_single_entity_coverage(row)


In [ ]:
print(result)
print(manager.partial_result_dir)

### 🗺️ process_inseason_bulk_extraction_parallel

In [ ]:
top25 = manager.sfd_list.head(50)
print(manager.sfd_list.head(50))

In [ ]:
from earthdaily.agriculture.core.logging_setup import setup_logging

# Initialize logging once at the start
setup_logging(
    log_dir="logs",
    log_level="DEBUG",
    log_to_console=False,  # ✅ Disable console output
    rotation="1 day",
    retention="30 days"
)

In [ ]:

top25 = manager.sfd_list.head(50)

results = extractor.process_entity_coverage_bulk_parallel(
    entity_list=top25,
    generate_report= True,
    params=None,          # or pass overrides here
    max_workers=20,        # adjust threads depending on API rate limits
    output_path=manager.output_result_dir,
    partial_frequency= 50,
    fail_safe= False,
    filter_column="crop.id",
    filter_value="OTHERS",
    filter_type="exclude", # filter type used to 'include' or 'exclude' row matching column and value filter
    merge_existing='auto',
    skip_export=False,
    prefix='coverage'
)


In [ ]:
# Get the clean DataFrame
results=results["results_df"]
print(results.columns)

## **Step 3b: Crop Coverage Filter Mode (historical_seasons)**

Test the `crop_coverage` filter mode which extracts coverage across multiple historical years.
Each entity's `start_date` / `end_date` month-day is combined with the `historical_seasons` years
to build the widest extraction window (smallest year start → largest year end).

**Per-entity override:** Each entity row can provide its own `historical_seasons` column
(list of years) to override the global value set in `setup_coverage_parameters()`.
Entities without the field fall back to the global parameter.

### 3b.1 Setup: Configure crop_coverage extractor

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

crop_cov_extractor = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

# Column mapping for platform data
column_mapping = {
    "crop": "crop.id",
    "start_date": "sowingDate",   # map to your DataFrame column for season start
    "end_date": "endDate"         # map to your DataFrame column for season end
}

crop_cov_extractor.setup_coverage_parameters(
    vegetation_index='NDVI',
    start_date='2025-07-01',
    end_date='2025-07-31',              # fallback date, overridden per entity in crop_coverage mode
    clear_cover_min=80,
    mask='auto',
    filter="crop_coverage",        # activate crop_coverage filter mode
    historical_seasons=[2022, 2024, 2025],  # years to span
    column_mapping=column_mapping,
    exclude_columns=[]
)

### 3b.2 Test single entity with crop_coverage

In [ ]:
import pandas as pd
from earthdaily.agriculture.reporting.viz_engine import crosstab_chart

# Test entity with start_date and end_date for crop_coverage
# Only July images will be kept for each historical year
# API fetches 2022-07-01 to 2025-07-31, then post-filters per year
test_entity_crop_cov = pd.Series({
    "id": "crop_cov_test_001",
    "geometry": "POLYGON((-58.9454 -13.7203, -58.9422 -13.7317, -58.9281 -13.7303, -58.9316 -13.7189, -58.9454 -13.7203))",

    "crop.id": "CORN"
})

result = crop_cov_extractor.process_single_entity_coverage(test_entity_crop_cov)

if result["data"] is not None:
    df = result["data"]
    print(f"Total images found: {len(df)}")
    print(f"Date range: {df['date'].min()} to {df['date'].max()}")

    # Heatmap: image count per month per year (should only show July)
    df["year"] = df["date"].str[:4]
    df["month"] = df["date"].str[5:7]
    crosstab_chart(df, row_col="year", col_col="month")
else:
    print(f"No data: {result['error']}")

### 3b.2b Test per-entity historical_seasons override

Each entity can define its own `historical_seasons` list, overriding the global parameter.
Entities without the column fall back to the global `[2022, 2024, 2025]` set in setup.

In [ ]:
# Per-entity historical_seasons override
# Entity A: uses its own [2023, 2024] instead of the global [2022, 2024, 2025]
# Entity B: no historical_seasons column -> falls back to global params

test_entity_custom_hs = pd.Series({
    "id": "crop_cov_custom_hs_001",
    "geometry": "POLYGON((-58.9454 -13.7203, -58.9422 -13.7317, -58.9281 -13.7303, -58.9316 -13.7189, -58.9454 -13.7203))",
    "crop.id": "CORN",
    "historical_seasons": [2023, 2024]  # per-entity override
})

result_custom = crop_cov_extractor.process_single_entity_coverage(test_entity_custom_hs)

if result_custom["data"] is not None:
    df_custom = result_custom["data"]
    print(f"Per-entity historical_seasons: Total images found: {len(df_custom)}")
    print(f"Date range: {df_custom['date'].min()} to {df_custom['date'].max()}")
    df_custom["year"] = df_custom["date"].str[:4]
    print(f"Years present: {sorted(df_custom['year'].unique())}")
    # Should only show 2023 and 2024, not 2022 or 2025
else:
    print(f"No data: {result_custom['error']}")

### 3b.3 Bulk extraction with crop_coverage

In [ ]:
# Ensure your entity list has start_date and end_date columns
# (mapped via column_mapping set in setup_coverage_parameters)
entities = manager.sfd_list.head(20)
print(f"Entities columns: {list(entities.columns)}")
print(entities.head())

In [ ]:
results_crop_cov = crop_cov_extractor.process_entity_coverage_bulk_parallel(
    entity_list=entities,
    max_workers=10,
    output_path=manager.output_result_dir,
    partial_frequency=50,
    fail_safe=False,
    skip_export=False,
    prefix="crop_coverage",
    generate_report=True
)

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import crosstab_chart

# Inspect results
df_crop_cov = results_crop_cov["results_df"]
print(f"Total rows: {len(df_crop_cov)}")
print(f"Columns: {list(df_crop_cov.columns)}")

if not df_crop_cov.empty:
    print(f"\nDate range: {df_crop_cov['date'].min()} to {df_crop_cov['date'].max()}")

    # Heatmap: image count per month per year
    df_crop_cov["year"] = df_crop_cov["date"].str[:4]
    df_crop_cov["month"] = df_crop_cov["date"].str[5:7]
    crosstab_chart(df_crop_cov, row_col="year", col_col="month")

## **🗃️ Step 4: Cache Testing**

Test the caching layer to verify:
1. First run populates the cache (all misses)
2. Second run serves from cache (all hits)
3. Partial cache scenario (mix of hits and misses)
4. Cache management utilities (`cache_info`, `clear_cache`)

### 4.1 Setup: Create a cache-enabled extractor

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

# Create a cache-enabled extractor
cache_extractor = CoverageExtractor(
    manager.bearer_token,
    manager.token_expiration,
    config={**manager.config, "use_cache": True, "cache_ttl_days": 7}
)

# Configure with same parameters as before
cache_extractor.setup_coverage_parameters(
    vegetation_index='NDVI',
    start_date="2025-01-01",
    clear_cover_min=80,
    use_specific_date=False,
    filter="duplicate",
    delay=3,
    mask='auto',
    partial_frequency=20,
    exclude_columns=[],
    column_mapping={"crop": "crop.id"}
)

# Clear any previous cache to start fresh
cache_extractor.clear_cache(params=cache_extractor.coverage_params)
print(f"\nCache directory: {cache_extractor.cache_dir}")
print(f"Cache TTL: {cache_extractor.cache_ttl_days} days")
print(f"Cache key columns: {cache_extractor.cache_key_columns}")

### 4.2 First run — all cache misses (populates cache)

In [ ]:
import time

# Use a small subset for testing
test_entities = manager.sfd_list.head(20)
print(f"Test entities: {len(test_entities)}")

# First run: everything should be fetched from API
start = time.time()
result_1 = cache_extractor.process_entity_coverage_bulk_parallel(
    entity_list=test_entities,
    max_workers=10,
    skip_export=True,
    prefix='cache_test',
    use_cache=True
)
elapsed_1 = time.time() - start

print(f"\n--- First Run (cold cache) ---")
print(f"Time: {elapsed_1:.2f}s")
print(f"Cache hits:   {result_1.get('cache_hit', 'N/A')}")
print(f"Cache misses: {result_1.get('cache_miss', 'N/A')}")
print(f"Results rows: {len(result_1['results_df'])}")

# Show cache state after first run
info = cache_extractor.cache_info(params=cache_extractor.coverage_params)
print(f"\nCache state: {info['records']} records in {info['path']}")

### 4.3 Second run — full cache hit (no API calls)

In [ ]:
# Second run: same entities — should be 100% cache hit, no API calls
start = time.time()
result_2 = cache_extractor.process_entity_coverage_bulk_parallel(
    entity_list=test_entities,
    max_workers=10,
    skip_export=True,
    prefix='cache_test',
    use_cache=True
)
elapsed_2 = time.time() - start

print(f"--- Second Run (warm cache) ---")
print(f"Time: {elapsed_2:.2f}s")
print(f"Cache hits:   {result_2.get('cache_hit', 'N/A')}")
print(f"Cache misses: {result_2.get('cache_miss', 'N/A')}")
print(f"Results rows: {len(result_2['results_df'])}")
print(f"\nSpeedup: {elapsed_1 / max(elapsed_2, 0.001):.1f}x faster")

### 4.4 Partial cache — mix of cached and new entities

In [ ]:
# Partial cache scenario: 20 entities already cached + 10 new ones
extended_entities = manager.sfd_list.head(30)  # first 20 cached, last 10 new
print(f"Extended entity set: {len(extended_entities)} (20 cached + 10 new)")

start = time.time()
result_3 = cache_extractor.process_entity_coverage_bulk_parallel(
    entity_list=extended_entities,
    max_workers=10,
    skip_export=True,
    prefix='cache_test',
    use_cache=True
)
elapsed_3 = time.time() - start

print(f"\n--- Third Run (partial cache) ---")
print(f"Time: {elapsed_3:.2f}s")
print(f"Cache hits:   {result_3.get('cache_hit', 'N/A')}")
print(f"Cache misses: {result_3.get('cache_miss', 'N/A')}")
print(f"Results rows: {len(result_3['results_df'])}")

# Verify results are consistent
print(f"\nCache now has: {cache_extractor.cache_info(params=cache_extractor.coverage_params)['records']} records")

### 4.5 Data consistency check

In [ ]:
import pandas as pd

# Compare first-run results vs second-run results (should be identical data)
df1 = result_1['results_df'].drop(columns=['_cached_at'], errors='ignore').sort_values(['id', 'image_id']).reset_index(drop=True)
df2 = result_2['results_df'].drop(columns=['_cached_at'], errors='ignore').sort_values(['id', 'image_id']).reset_index(drop=True)

# Check shape match
print(f"Run 1 shape: {df1.shape}")
print(f"Run 2 shape: {df2.shape}")
print(f"Shapes match: {df1.shape == df2.shape}")

# Check value equality (NaN-safe: NaN == NaN is treated as True)
if df1.shape == df2.shape:
    comparison = df1.fillna('__NAN__').eq(df2.fillna('__NAN__'))
    mismatches = (~comparison).sum().sum()
    print(f"Cell mismatches: {mismatches}")
    if mismatches == 0:
        print("Data consistency: PASSED")
    else:
        print("Data consistency: FAILED — investigate mismatched columns")
        print((~comparison).sum()[lambda x: x > 0])

### 4.6 Cache management utilities

In [ ]:
# Inspect cache details
info = cache_extractor.cache_info(params=cache_extractor.coverage_params)
print("Cache Info:")
for k, v in info.items():
    print(f"  {k}: {v}")

# Peek at cached parquet
cached_df = pd.read_parquet(info['path'])
print(f"\nCached DataFrame: {cached_df.shape}")
print(f"Columns: {list(cached_df.columns)}")
print(f"\nSample cached rows:")
display(cached_df.head(3))

In [ ]:
# Use_cache can also be passed per-call to override the instance setting
# Here we disable cache for a single run even though the extractor has use_cache=True
result_no_cache = cache_extractor.process_entity_coverage_bulk_parallel(
    entity_list=manager.sfd_list.head(5),
    max_workers=5,
    skip_export=True,
    prefix='no_cache_test',
    use_cache=False  # Override: bypass cache for this run
)
print(f"Results without cache: {len(result_no_cache['results_df'])} rows")
print(f"cache_hit key present: {'cache_hit' in result_no_cache}")

In [ ]:
# Cleanup: clear cache when done testing
cache_extractor.clear_cache(params=cache_extractor.coverage_params)
print("Cache cleared.")
print(cache_extractor.cache_info(params=cache_extractor.coverage_params))